# 03 · Narrative AI demo → `alerts_ai_summaries`

This skeleton reads `alerts` and writes a demo table `alerts_ai_summaries` with a
deterministic (non-AI) formatter. In Fabric, replace the formatter with your AI Function
`SUMMARIZE_ALERT(province, organism, antibiotic, year, tau, theta_hat, pr_exceed_tau, is_stable_alert, reason, n_tested)`.

In [ ]:
from pyspark.sql import functions as F, types as T
alerts = spark.table('alerts')
display(alerts.limit(10))

## Placeholder summarizer (deterministic)

In [ ]:
def _format_summary(province, organism, antibiotic, year, tau, theta_hat, pr_exceed_tau, is_stable, reason, n):
    status = "Stable alert" if is_stable else "No stable alert"
    title = f"{province}: {organism}–{antibiotic} ({year}) — {status}"
    b1 = f"τ={tau:.2f}, θ̂={theta_hat:.2f}, Pr(θ>τ)={pr_exceed_tau:.2f}"
    b2 = f"Rule: {reason}; n={n}"
    action = ("Review empiric policy and consider restriction." if is_stable else "Continue monitoring; no policy change.")
    sms = f"{province} {organism}/{antibiotic} {year}: {status} (Pr>{tau:.2f}={pr_exceed_tau:.2f}, n={n})."
    return (title, [b1, b2, action], sms[:160])

schema = T.StructType([
    T.StructField('title', T.StringType(), False),
    T.StructField('bullets', T.ArrayType(T.StringType()), False),
    T.StructField('sms', T.StringType(), False)
])

format_udf = F.udf(_format_summary, schema)

## Build `alerts_ai_summaries`

In [ ]:
enriched = (alerts
  .withColumn('row_key', F.concat_ws('|', 'province','organism','antibiotic','specimen', F.col('year').cast('string')))
  .withColumn('fmt', format_udf('province','organism','antibiotic','year','tau','theta_hat','pr_exceed_tau','is_stable_alert','reason','n_tested'))
  .withColumn('title', F.col('fmt.title'))
  .withColumn('bullets', F.col('fmt.bullets'))
  .withColumn('sms', F.col('fmt.sms'))
  .withColumn('created_at', F.current_timestamp())
  .select('row_key','title','bullets','sms','created_at')
)

spark.sql('DROP TABLE IF EXISTS alerts_ai_summaries')
enriched.write.mode('overwrite').format('delta').saveAsTable('alerts_ai_summaries')
spark.sql('REFRESH TABLE alerts_ai_summaries')
display(spark.table('alerts_ai_summaries').orderBy('row_key').limit(10))

### Swap-in (later) for Fabric AI Function
Replace the `_format_summary` UDF with a call to your AI Function and keep the same output schema.